In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import matplotlib.pyplot as plt

transform = transforms.ToTensor()

train_set = datasets.CIFAR10(root="./data", train=True,  download=True, transform=transform)
test_set  = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False)

In [ ]:
images, labels = next(iter(train_loader))

print("images:", images.shape)
print("labels:", labels.shape)

print("train_set:", len(train_set), "| test_set:", len(test_set))
print("train_loader:", len(train_loader))

print("10 first labels:", labels[:10])
print("Class names:", [train_set.classes[i] for i in labels[:10]])

fig, axes = plt.subplots(1, 8, figsize=(14, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(images[i].permute(1, 2, 0), cmap="gray")
    ax.set_title(train_set.classes[labels[i]], fontsize=9)
    ax.axis("off")
plt.show()

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.flatten = nn.Flatten()
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, 512)
        self.fc3 = nn.Linear(512, 256)
        self.fc4 = nn.Linear(256,10)
    
        # Initialize He for Linear layers
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_uniform_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.fc4(x)
        return x

input_dim = 3 * 32 * 32
model = MLP(input_dim)
print(model)

out = model(images)
print("output:", out.shape)

total = sum(p.numel() for p in model.parameters())
print("Total params:", total)

In [ ]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("device:", device)

torch.manual_seed(42)
model = MLP(input_dim).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)  # forward
        loss = criterion(logits, labels) # compute loss

        optimizer.zero_grad()   # clear gradient
        loss.backward()         # backprop
        optimizer.step()        # update params

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        n += images.size(0)
    return total_loss / n, correct / n

def evaluate(model, loader, criterion, device):
    model.eval()                                   # chế độ đánh giá
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():                          # không tính gradient
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * images.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            n += images.size(0)
    return total_loss / n, correct / n

In [ ]:
epochs = 10
for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)
    print(f"Epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | "
          f"test loss {te_loss:.4f} acc {te_acc:.4f}")